<a href="https://colab.research.google.com/github/rahelDK/RAG/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#INSTALL & IMPORT DEPENDENCIES

In [ ]:
!pip -q install pdfplumber sentence-transformers faiss-cpu transformers accelerate
import pdfplumber
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 983.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 742.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 24.4 MB/s eta 0:00:00


In [ ]:
from google.colab import files, drive
#uploaded = files.upload()
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

pdf_path = "/content/drive/MyDrive/10-K-2025-Apple.pdf"

#Extract Text From PDF

In [ ]:
def extract_text_from_pdf(pdf_file_path: str):
    pages_metadata = []
    full_text = []

    with pdfplumber.open(pdf_file_path) as pdf:
        for i, page in enumerate(pdf.pages, start=1):
            text = page.extract_text() or ""
            text = " ".join(text.split())
            pages_metadata.append({"page_number": i, "text": text})
            full_text.append(text)

    return "\n".join(full_text), pages_metadata


raw_text, pages_metadata = extract_text_from_pdf(pdf_path)
print("Pages extracted:", len(pages_metadata))
print("Sample (first 500 chars):", raw_text[:500])

Pages extracted: 80
Sample (first 500 chars): UNITED STATES SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549 FORM 10-K (Mark One) ☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 For the fiscal year ended September 27, 2025 or ☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 For the transition period from to . Commission File Number: 001-36743 Apple Inc. (Exact name of Registrant as specified in its charter) California 94-2404110 (State or other jurisdi


#Chunk Text

In [ ]:
def chunk_text_with_metadata(pages_data, chunk_size=200, overlap=50):
    chunks = []
    step = chunk_size - overlap

    for page in pages_data:
        words = page["text"].split()
        if not words:
            continue

        for start in range(0, len(words), step):
            end = start + chunk_size
            chunk_words = words[start:end]
            if not chunk_words:
                continue

            chunks.append({
                "text": " ".join(chunk_words),
                "page_number": page["page_number"]
            })

    return chunks


text_chunks = chunk_text_with_metadata(pages_metadata, chunk_size=200, overlap=50)
print("Total chunks:", len(text_chunks))
print("Example chunk:", text_chunks[0]["page_number"], text_chunks[0]["text"][:250])

Total chunks: 322
Example chunk: 1 UNITED STATES SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549 FORM 10-K (Mark One) ☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 For the fiscal year ended September 27, 2025 or ☐ TRANSITION REPORT PU


#CREATE EMBEDDINGS

In [ ]:
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

chunk_texts = [c["text"] for c in text_chunks]
chunk_embeddings = embedder.encode(chunk_texts, show_progress_bar=True).astype("float32")


faiss.normalize_L2(chunk_embeddings)

dimension = chunk_embeddings.shape[1]
print("Embedding dimension:", dimension)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Embedding dimension: 384


#Build FAISS index

In [ ]:
index = faiss.IndexFlatL2(dimension)
index.add(chunk_embeddings)

print("FAISS index size:", index.ntotal)

FAISS index size: 322


#Return top-k relevant chunks

In [ ]:
def retrieve_relevant_chunks(query: str, top_k: int = 3):
    q_emb = embedder.encode([query]).astype("float32")
    faiss.normalize_L2(q_emb)

    distances, indices = index.search(q_emb, top_k)
    results = []
    for rank, idx in enumerate(indices[0]):
        if idx == -1:
            continue
        results.append({
            "rank": rank + 1,
            "page_number": text_chunks[idx]["page_number"],
            "text": text_chunks[idx]["text"],
            "distance": float(distances[0][rank])
        })
    return results

#Load the generator model (FLAN-T5) + define RAG QA

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "google/flan-t5-base"

tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)

def rag_qa_pipeline(
    query: str,
    top_k: int = 3,
    max_new_tokens: int = 200,
    temperature: float = 0.3,
    top_p: float = 0.9,
    top_k_gen: int = 50
):
    retrieved = retrieve_relevant_chunks(query, top_k=top_k)
    context = "\n\n".join(
        [f"(Page {r['page_number']}) {r['text']}" for r in retrieved]
    )

    prompt = f"""You are answering ONLY using the context.
If the answer is not in the context, say: "I don't have enough information in the provided document."

Context:
{context}

Question: {query}
Answer:"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(device)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k_gen
    )

    answer = tokenizer.decode(output_ids[0], skip_special_tokens=True)


    cited_pages = sorted(set([r["page_number"] for r in retrieved]))
    return answer, retrieved, cited_pages

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

#TEST THE COMPLETE RAG SYSTEM

In [ ]:
question = "What are Apple’s total net sales for the fiscal year?"
answer, retrieved, cited_pages = rag_qa_pipeline(question, top_k=4)

print("ANSWER:\n", answer)
print("\nCITED PAGES:", cited_pages)

ANSWER:
 $268654,952.66,513

CITED PAGES: [26, 32, 39, 50]


In [ ]:
question = "What subscription-based services does Apple offer?"
answer, retrieved, cited_pages = rag_qa_pipeline(question, top_k=4)

print("ANSWER:\n", answer)
print("\nCITED PAGES:", cited_pages)

ANSWER:
 devices can listen simultaneously on in. television station live music television service live television Apple Fitness Plus app subscription apps to stream. fitness news programs app music and programming Apple videos are on to provide the information to its television client as it moves the content as desired in many applications by streaming multiple channels all and having services of physical content streams play the same digital data simultaneously requiring different device service. You are presented content from programs released on iPhone in select products which

CITED PAGES: [4, 5, 15]


In [ ]:
question = "What risks does Apple disclose regarding competition?"
answer, retrieved, cited_pages = rag_qa_pipeline(question, top_k=4)

print("ANSWER:\n", answer)
print("\nCITED PAGES:", cited_pages)

ANSWER:
 decline in growth and lowered profit levels over time while operating at lower energy costs/linquy than when operating internally instead; failure on competitive performance as such

CITED PAGES: [5, 10, 12, 17]
